# Classification Primer — Guided Practice

In the regression primer, you predicted a numerical house price. Here the target
is a **category**: can the words in a baby-product review help us predict whether
its star rating is positive or negative?

By the end, you should be able to:

- prepare text and create a clearly defined classification target;
- turn reviews into word-count features without learning from held-out data;
- construct, fit, and interpret a logistic-regression classifier;
- distinguish predicted labels from estimated probabilities;
- compare two vocabularies with a simple baseline and inspect their errors;
- choose an approach using validation data, then assess it once on a test set.



## How to work through this notebook

Complete Exercises 1–5 in order. Worked examples introduce new techniques on
tiny inputs. In **Your turn**, write complete blocks of code: prepare tables,
create and fit models, predict, organise results, and draw plots. The cells
contain comments rather than lines with single blanks.

Required object names are stated because later cells reuse them. You choose
intermediate names and how to organise your implementation. Try each task
before opening its optional hint. Written explanations are part of the work.

Run a checkpoint after completing the preceding task. An unfinished checkpoint
deliberately stops with a message. Once the core work is complete, restart the
kernel and run all cells in order. Extensions are optional.

We use **all eligible reviews**, with no fixed-size sample. Leave the test
reviews aside until Exercise 5. Keep the supplied split and model settings fixed
so the comparison answers a clear question.


## Setup — reviews and a proxy for sentiment

Keep **amazon_baby.csv beside this notebook**. In Colab, upload the CSV into
the session's files. You need pandas, numpy, matplotlib, and scikit-learn.
The CSV is supplied with this assignment; no online service or account is needed.

A row contains a product name, review text, and rating. We define **1–2 stars
as negative (−1)** and **4–5 stars as positive (+1)**. We leave out 3-star
reviews to keep this first task binary. This does not mean all 3-star texts
express neutral feelings. Ratings are a useful **proxy label**, not a direct
measurement of every sentence's sentiment.

Use review text as input. The rating creates the target and must not also be
an input feature. Product names are available for inspection, not modelling.

Counting words and fitting on the full dataset can take a few minutes on a laptop. Keep the large word-count matrices sparse.


In [ ]:
from pathlib import Path
from time import perf_counter
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

possible_paths = [Path("amazon_baby.csv"), Path("Primer Assignments") / "amazon_baby.csv"]
data_path = next((path for path in possible_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Put amazon_baby.csv beside this notebook, then run this cell again.")
reviews = pd.read_csv(data_path)
print("Source records:", len(reviews))
display(reviews.head())


## Exercise 1 — Prepare the target and inspect the balance

**Your turn — audit the input.** Display the number of missing values in
each column and a table counting ratings from 1 to 5. Then find two nonmissing
example reviews with different ratings and display their text and rating.
Explain whether their labels seem reasonable; a mixed review is also useful.

Keep reviews unchanged so you can compare the prepared data with the source.


In [ ]:
# Your code: inspect missing values, count ratings, and show two contrasting examples.


**Worked example — a small text-cleaning function.** This deliberately simple
function deletes ASCII punctuation. It takes a string, so replace missing
values **before** calling it. Series.apply applies a function to each element.

Deleting punctuation has costs: "don't" becomes "dont", and "easy-to-use"
becomes "easytouse". We keep this rule fixed to study its limitations.
Cleaning text is not the same as understanding it.


In [ ]:
def remove_punctuation(text):
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator)

tiny_texts = pd.Series(["Easy-to-use! I don't regret it.", None])
tiny_cleaned = tiny_texts.fillna("").apply(remove_punctuation).str.lower().str.strip()
display(pd.DataFrame({"Original": tiny_texts, "Cleaned": tiny_cleaned}))


**Your turn — implement the full preparation.** Create a copy of reviews named
**prepared_reviews**. Preserve the original review and rating columns.

1. Create **review_clean** by filling missing review text with an empty string,
   applying remove_punctuation, converting to lower case, and stripping
   whitespace from the ends.
2. Keep only ratings 1, 2, 4, and 5. Remove rows whose cleaned review is empty:
   there is no text to learn from. Do not discard a review because its product
   name is missing.
3. Create **sentiment**, equal to −1 for ratings 1–2 and +1 for ratings 4–5.
4. Print how many records remain. Construct **class_counts**, a Series with
   counts indexed by −1 and +1. Display counts and percentages, then draw a
   labelled bar chart of the two classes.

Do not take a random sample or balance the classes by deleting positive reviews.
The imbalance is part of the prediction problem.

<details><summary>Optional preparation hints</summary>

Start from reviews.copy(). Series.isin([1, 2, 4, 5]) makes a selection mask;
select rows with .loc[mask].copy(). np.where(condition, value_if_true,
value_if_false) can create the two labels. value_counts().sort_index()
counts labels in order. A Series can be plotted using .plot.bar(ax=ax).

</details>


In [ ]:
# Your code: prepare the complete eligible dataset, create class_counts, and plot the class balance.


In [ ]:
assert "prepared_reviews" in globals() and "class_counts" in globals(), "Prepare the reviews and count the two classes first."
expected_clean = reviews["review"].fillna("").apply(remove_punctuation).str.lower().str.strip()
expected_index = reviews.index[reviews["rating"].isin([1, 2, 4, 5]) & expected_clean.ne("")]
assert prepared_reviews.index.equals(expected_index), "Keep every eligible review, in its original order, with no sample cap."
assert prepared_reviews["review_clean"].equals(expected_clean.loc[expected_index]), "Check missing text, punctuation, case, and whitespace."
expected_labels = np.where(prepared_reviews["rating"].ge(4), 1, -1)
assert np.array_equal(prepared_reviews["sentiment"], expected_labels), "Check the rating-to-sentiment rule."
assert class_counts.sort_index().equals(prepared_reviews["sentiment"].value_counts().sort_index()), "Count the prepared labels."
print("Checkpoint passed: the full eligible dataset and its two labels are ready.")


**Explain:** Which class is more common? Roughly what accuracy would a rule
predicting that class for every review achieve? Would that rule identify
any negative reviews? Explain why a high-looking accuracy can be misleading.
Also explain why rating must not be used as an input.


_Your answer:_


## Exercise 2 — From text to numerical features

**Worked example — bag of words.** CountVectorizer learns a vocabulary from
training text and counts each word. Each row is a review, each column a token
(a word-like unit), and each entry its count. The default lowercases text and
ignores single-character tokens. Repeated words increase their counts.

The result is a **sparse matrix**: it stores nonzero counts efficiently. We turn
only this tiny example into a dense table. Keep the real review matrices sparse;
do not call .toarray() or .todense() on the full dataset.


In [ ]:
toy_reviews = [
    "we love this easy product",
    "great product we love it",
    "we hate this broken product",
    "awful product we hate it",
    "we love love this",
    "we hate hate this",
]
toy_labels = np.array([1, 1, -1, -1, 1, -1])
toy_vectorizer = CountVectorizer()
X_toy = toy_vectorizer.fit_transform(toy_reviews)
toy_words = toy_vectorizer.get_feature_names_out()
display(pd.DataFrame(X_toy.toarray(), columns=toy_words, index=toy_reviews))

toy_new_reviews = ["we love bananas", "bananas"]
X_toy_new = toy_vectorizer.transform(toy_new_reviews)
display(pd.DataFrame(X_toy_new.toarray(), columns=toy_words, index=toy_new_reviews))


**Read the tables:** Why is bananas absent from the columns? What happened to
the second new review? Would "we love this product" and "this product we love"
have different count vectors? State one meaning the representation loses.


_Your answer:_


### Three groups, with matching review texts kept together

We reuse the regression workflow: approximately **60% training**, **20%
validation**, and **20% test**. All eligible records are retained.

Some review texts repeat. A random row split could put a copy of a review in
both training and evaluation. The supplied GroupShuffleSplit assigns groups
defined by the same cleaned text (ignoring repeated whitespace) together.
Its test_size refers to groups, so row percentages are approximate. This new
splitting method is supplied; you are not expected to implement it.

The split does **not** separate products, authors, near-duplicate texts, or time
periods. It assesses other reviews in a similar dataset, not performance on
entirely new products or future reviews. Missing author/time information also
limits what we can check.

We inspect class balance in training and validation only. Test labels are not
used to compare models or choose settings.


In [ ]:
prepared_reviews["review_group"] = prepared_reviews["review_clean"].str.split().str.join(" ")
first_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=44)
development_positions, test_positions = next(
    first_split.split(prepared_reviews, groups=prepared_reviews["review_group"])
)
development_data = prepared_reviews.iloc[development_positions].copy()
test_data = prepared_reviews.iloc[test_positions].copy()

second_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=44)
train_positions, validation_positions = next(
    second_split.split(development_data, groups=development_data["review_group"])
)
train_data = development_data.iloc[train_positions].copy()
validation_data = development_data.iloc[validation_positions].copy()
y_train = train_data["sentiment"]
y_validation = validation_data["sentiment"]

print("Training / validation / reserved test rows:", len(train_data), len(validation_data), len(test_data))
print("Repeated text records beyond the first:", prepared_reviews["review_group"].duplicated().sum())
display(pd.DataFrame({
    "Training share": y_train.value_counts(normalize=True),
    "Validation share": y_validation.value_counts(normalize=True),
}).sort_index())

assert len(train_data) + len(validation_data) + len(test_data) == len(prepared_reviews)
for left, right in [(train_data, validation_data), (train_data, test_data), (validation_data, test_data)]:
    assert set(left["review_group"]).isdisjoint(right["review_group"])


**Your turn — build the real feature matrices.** Create a new CountVectorizer
named **full_vectorizer**, using its default settings. Learn its vocabulary
and create **X_train** from train_data["review_clean"]. Create
**X_validation** using the same fitted vocabulary and validation review text.

Pass a **one-dimensional Series of strings**, using single brackets to select the text column. The vectorizer creates the two-dimensional feature matrix for the classifier.

Print both matrix shapes, the vocabulary size, and the number of stored
nonzero entries in X_train (its .nnz attribute). Explain what the two dimensions
mean. Use every training and validation review and keep both matrices sparse.
Here "full" means the complete vocabulary learned by this tokenizer from
training reviews, rather than the 20-word vocabulary used later.

<details><summary>Optional API hint</summary>

fit_transform learns the vocabulary and produces counts. transform only
produces counts using an already learned vocabulary. The toy example shows
both operations. get_feature_names_out() returns the learned column names.

</details>


In [ ]:
# Your code: create the vectorizer, learn training counts, transform validation text, and report their dimensions.


In [ ]:
assert all(name in globals() for name in ["full_vectorizer", "X_train", "X_validation"]), "Create the vectorizer and both sparse matrices first."
assert X_train.shape[0] == len(train_data) and X_validation.shape[0] == len(validation_data), "Use the assigned rows."
assert X_train.shape[1] == X_validation.shape[1] == len(full_vectorizer.get_feature_names_out()), "Both groups must use the same vocabulary."
assert full_vectorizer.get_params()["vocabulary"] is None, "Learn the first model's vocabulary from training text."
assert (X_train != full_vectorizer.transform(train_data["review_clean"])).nnz == 0, "Check the training matrix and its row order."
assert (X_validation != full_vectorizer.transform(validation_data["review_clean"])).nnz == 0, "Transform validation text using the fitted training vectorizer."
print("Checkpoint passed: consistent word-count columns for training and validation.")


**Explain:** What does each dimension of X_train represent? Why must we avoid
fitting a new vectorizer on validation text? Why would fitting on the entire
dataset before splitting also be a problem?


_Your answer:_


## Exercise 3 — Learn a classifier and a baseline

**Worked example — logistic regression.** Despite its name, LogisticRegression
is a classification model. It combines word counts using learned coefficients
and converts the resulting score into an estimated probability between 0 and 1.
No logistic-function algebra is required here.

- .fit(counts, labels) learns from labelled training examples.
- .predict(counts) returns a label.
- .predict_proba(counts) returns one column per label, in .classes_ order.

For this binary model with classes [−1, +1], a positive coefficient pushes
the score towards +1, holding other counts fixed. A coefficient is not itself
a probability. We fix solver="liblinear", C=1.0, max_iter=1000, and
random_state=44 in this primer. The solver is the fitting routine; C controls
coefficient regularisation. Tuning these settings is outside the core work.


In [ ]:
toy_model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=44)
toy_model.fit(X_toy, toy_labels)
positive_column_toy = list(toy_model.classes_).index(1)
toy_probabilities = toy_model.predict_proba(X_toy_new)[:, positive_column_toy]
print("Probability columns correspond to:", toy_model.classes_)
display(pd.DataFrame({
    "Text": toy_new_reviews,
    "Predicted label": toy_model.predict(X_toy_new),
    "Estimated P(+1)": toy_probabilities,
}))


**Your turn — fit and apply the full-vocabulary model.** Create **full_model**
with the fixed LogisticRegression settings above. Fit it on X_train and y_train.
Measure just the .fit call with perf_counter and save **full_fit_seconds**.

Predict validation labels as **full_validation_predictions**. Separately time
just that .predict call as **full_predict_seconds**. Then obtain **full_p_positive**,
a one-dimensional array containing each validation review's estimated probability
of +1. Find the correct probability column from full_model.classes_.

Display labels and positive probabilities for the first five validation rows,
together with their ratings. Keep the row order consistent.

<details><summary>Optional timing and probability hints</summary>

Save start = perf_counter() immediately before the operation and subtract it
from perf_counter() immediately after. The toy example shows how to find the
probability column. An array slice [:5] selects the first five entries.

</details>


In [ ]:
# Your code: construct full_model, fit and time it, predict labels and probabilities, and display a small result table.


In [ ]:
required = ["full_model", "full_validation_predictions", "full_p_positive", "full_fit_seconds", "full_predict_seconds"]
assert all(name in globals() for name in required), "Complete model fitting, timed prediction, and positive-class probabilities."
assert np.array_equal(full_model.classes_, [-1, 1]), "Fit using the requested sentiment labels."
assert np.array_equal(full_validation_predictions, full_model.predict(X_validation)), "Predict the validation reviews in their original order."
positive_column = list(full_model.classes_).index(1)
assert np.asarray(full_p_positive).shape == (len(validation_data),), "Keep one positive-class probability per review."
assert np.allclose(full_p_positive, full_model.predict_proba(X_validation)[:, positive_column]), "Check the probability column."
assert full_fit_seconds >= 0 and full_predict_seconds >= 0, "Store elapsed durations in seconds."
print("Checkpoint passed: labels and estimated positive probabilities are aligned.")


**Worked example — the majority-class baseline.** DummyClassifier with
strategy="most_frequent" learns only the most common training label and predicts
it for everyone. Its API matches the other classifiers.

Accuracy is the proportion of correct labels: correct predictions / all
predictions. It is not an average of the model's predicted probabilities.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_validation_predictions = baseline.predict(X_validation)
baseline_validation_accuracy = accuracy_score(y_validation, baseline_validation_predictions)
print("Baseline's predicted class:", baseline_validation_predictions[0])
print(f"Baseline validation accuracy: {baseline_validation_accuracy:.2%}")


**Your turn — calculate and organise performance.** Predict training labels
with full_model and calculate **full_training_accuracy**. Calculate
**full_validation_accuracy** from the validation predictions already obtained.

Build and display **initial_comparison**, a two-row DataFrame with the baseline
and full-vocabulary model, showing training and validation accuracy for each.
Also print the validation improvement over the baseline in **percentage points**.
For example, 0.90 − 0.85 = 0.05 is an improvement of 5 percentage points.


In [ ]:
# Your code: calculate both models' training/validation accuracy, build initial_comparison, and report the improvement.


**Compare:** Does learning from words help compared with the baseline? Quote
validation values. Is training accuracy enough to assess that? What does a
probability of 0.95 mean, and does it guarantee that a review is positive?


_Your answer:_


## Exercise 4 — Interpret words, probabilities, and mistakes

**Your turn — connect word names to coefficients.** Construct **word_weights**,
a DataFrame containing columns **word** and **coefficient**, with one row per
token. Pair full_vectorizer.get_feature_names_out() with full_model.coef_[0]
in their existing order. Display the 10 largest and 10 smallest coefficients.

Draw two horizontal bar charts, one for each group, with readable word labels,
titles, and a coefficient axis. Students in the same course may obtain
unintuitive words; interpret the fitted association rather than invent a
linguistic rule. Large coefficients on rare tokens deserve caution.

<details><summary>Optional table and plotting hints</summary>

DataFrame.nlargest(10, "coefficient") and .nsmallest(10, "coefficient")
select extreme values. ax.barh(words, coefficients) makes horizontal bars.
fig, axes = plt.subplots(1, 2, ...) gives two axes for related charts.

</details>


In [ ]:
# Your code: build word_weights, show both extremes, and construct two labelled coefficient plots.


**Your turn — rank estimated sentiment.** Make **validation_predictions_table**
by copying the name, review, rating, and sentiment columns of validation_data.
Add columns **predicted_sentiment**, **p_positive**, and **review_preview**.
The preview should contain the first 180 characters of the original review.
Attach prediction arrays positionally so they stay with the correct rows.

Display the five rows with the highest and five with the lowest p_positive,
showing ratings, labels, probabilities, and previews. Then display the first
three misclassified validation reviews in their existing order. Read their
**complete** text when explaining an error; a preview may hide the key sentence.

Highest and lowest probabilities mean the model's strongest estimated
sentiments, not reviews we have independently proved to be the most extreme.

<details><summary>Optional selection hints</summary>

Use .copy() before adding columns. .str.slice(0, 180) makes text previews.
A Boolean mask comparing sentiment and predicted_sentiment selects errors.
.head(3) selects the first three rows, with no cherry-picking.

</details>


In [ ]:
# Your code: assemble the aligned prediction table, rank both probability extremes, and inspect three complete errors.


**Interpret:** Give one positive-coefficient word and one negative-coefficient
word and explain their signs. Does a positive coefficient guarantee a positive
label whenever that word occurs? Choose one of the first three displayed errors,
identify its row, summarise its text, and suggest a plausible reason for the
error. Separate what you observed from what you hypothesise.


_Your answer:_


### Accuracy can hide missed negative reviews

**Worked example — reading a confusion matrix.** Rows are actual labels;
columns are predicted labels. We explicitly order both as **[−1, +1]**:

| | Predicted −1 | Predicted +1 |
|---|---|---|
| Actual −1 | Correctly detected negative | Missed negative |
| Actual +1 | Positive flagged as negative | Correctly detected positive |

**Negative-class recall** asks: of all actually negative reviews, what share
did we detect? It is the first cell / the sum of the first row. Calling
recall_score(..., pos_label=-1) makes −1 the class of interest for that metric.
The dataset labels and probability-column order stay unchanged.


In [ ]:
example_actual = np.array([-1, -1, -1, 1, 1, 1])
example_predicted = np.array([-1, 1, 1, -1, 1, 1])
example_matrix = confusion_matrix(example_actual, example_predicted, labels=[-1, 1])
display(pd.DataFrame(example_matrix, index=["Actual −1", "Actual +1"], columns=["Predicted −1", "Predicted +1"]))
print(f"Negative recall = 1/3 = {recall_score(example_actual, example_predicted, pos_label=-1):.1%}")


**Your turn — assess the real mistakes.** Calculate **full_confusion** with
labels=[-1, 1]. Display a labelled matrix using ConfusionMatrixDisplay (or your
own labelled plot). Calculate **full_negative_recall** using pos_label=-1.
Calculate the baseline's negative recall too, and display both values.

Print how many negative reviews the full model missed and how many positive
reviews it incorrectly flagged as negative. Use the stated row/column order.

<details><summary>Optional plotting hint</summary>

ConfusionMatrixDisplay(confusion_matrix=matrix,
display_labels=["Negative (−1)", "Positive (+1)"]).plot(values_format="d")
displays counts. Keep axis labels and a title explaining which group is shown.

</details>


In [ ]:
# Your code: calculate and plot full_confusion, compare negative recall, and count the two types of errors.


In [ ]:
required = ["word_weights", "validation_predictions_table", "full_confusion", "full_negative_recall"]
assert all(name in globals() for name in required), "Finish the word table, review inspection, and error analysis."
assert word_weights["word"].tolist() == full_vectorizer.get_feature_names_out().tolist(), "Keep the original word/coefficient alignment."
assert np.allclose(word_weights["coefficient"], full_model.coef_[0]), "Pair every word with its fitted coefficient."
assert validation_predictions_table.index.equals(validation_data.index), "Keep review rows and predictions aligned."
assert np.allclose(validation_predictions_table["p_positive"], full_p_positive), "Attach the matching probabilities."
assert np.array_equal(full_confusion, confusion_matrix(y_validation, full_validation_predictions, labels=[-1, 1])), "Check matrix label order."
assert np.isclose(full_negative_recall, recall_score(y_validation, full_validation_predictions, pos_label=-1)), "Calculate recall for the negative class."
print("Checkpoint passed: word weights, review rows, and error counts are aligned.")


**Explain:** Quote the validation accuracy and negative recall. How many
negative reviews were missed? If a shop used this model to flag complaints
for staff, which error would hide a complaint, and which would create extra
reading work? Why is the majority baseline inadequate for that use?


_Your answer:_


## Exercise 5 — Run a second experiment, choose, and test

Can a much smaller vocabulary retain useful predictive information? We fix
the following **20-word vocabulary before looking at validation results**.
It contains possible positive and negative cues, along with ordinary words.
The name significant_words is a convenient variable name, not a claim of
statistical significance.

**Your turn — implement the complete restricted-vocabulary workflow.**
Construct **limited_vectorizer** using CountVectorizer(vocabulary=significant_words).
Create its training and validation count matrices, then create and fit
**limited_model** with the same logistic-regression settings and labels.

Obtain **limited_validation_predictions**, **limited_training_accuracy**,
**limited_validation_accuracy**, and **limited_negative_recall**.
Measure **limited_fit_seconds** around .fit only and **limited_predict_seconds**
around validation .predict only, as for the full model. Display the coefficient
for every one of the 20 words, paired using get_feature_names_out().

Write the entire block yourself. Use the **same rows and ordering** as the
first model. Do not reuse its count matrix: the feature columns have changed.
A fixed vocabulary is allowed because this list was supplied in advance,
rather than selected by inspecting test reviews.


In [ ]:
significant_words = [
    "love", "great", "easy", "old", "little", "perfect", "loves", "well", "able", "car",
    "broke", "less", "even", "waste", "disappointed", "work", "product", "money", "would", "return",
]


In [ ]:
# Your code: implement the full 20-word experiment, calculate scores and timings, and display all word coefficients.


**Your turn — make the decision table and plot.** Construct
**validation_comparison**, with rows for "baseline", "full", and "limited".
Include training accuracy, validation accuracy, negative-class validation recall,
and vocabulary size. Add fit and validation label-prediction times in seconds
for the two logistic models; use a missing value for baseline timing.

Construct a grouped bar chart comparing validation accuracy and negative recall
for the three approaches. Label the scale as a proportion from 0 to 1, or
consistently convert all scores to percentages.

Timings exclude text vectorisation, probability prediction, and plotting.
They are single-run measurements on your computer, not a controlled speed
benchmark. Do not repeatedly refit merely to obtain an attractive time.


In [ ]:
# Your code: assemble validation_comparison from existing results and plot the two validation metrics.


In [ ]:
required = ["limited_vectorizer", "limited_model", "limited_validation_predictions", "limited_validation_accuracy", "limited_training_accuracy", "limited_negative_recall", "validation_comparison"]
assert all(name in globals() for name in required), "Finish the 20-word experiment and decision table first."
assert set(limited_vectorizer.get_feature_names_out()) == set(significant_words), "Use exactly the supplied 20-word vocabulary."
expected_limited = limited_model.predict(limited_vectorizer.transform(validation_data["review_clean"]))
assert np.array_equal(limited_validation_predictions, expected_limited), "Keep limited predictions aligned with the same validation rows."
assert np.isclose(limited_validation_accuracy, accuracy_score(y_validation, expected_limited)), "Check limited-model validation accuracy."
assert np.isclose(limited_negative_recall, recall_score(y_validation, expected_limited, pos_label=-1)), "Check the negative-class recall."
assert len(validation_comparison) == 3, "Compare the baseline and both logistic models."
print("Checkpoint passed: both vocabularies were evaluated on the same validation reviews.")


**Decide before testing:** Compare both vocabularies using validation accuracy,
negative recall, vocabulary size, and the measured times. What did the smaller
vocabulary gain and lose? Choose one approach and justify the tradeoff.
If both accuracy and negative recall favour one model, say so.

Then assign **chosen_name** to "baseline", "full", or "limited" in the next cell.
Keep the standard prediction rule; threshold experiments are optional later.


_Your answer:_


In [ ]:
# Your code: assign chosen_name after explaining your validation-based decision.


### Final assessment — a familiar refit, now including text preprocessing

**Your turn — construct the final workflow.** The regression primer supplied
the final refit; this time you implement it. A pipeline ensures that learning
a vocabulary and fitting a classifier happen together.

For a tiny syntax example,
make_pipeline(CountVectorizer(), LogisticRegression(solver="liblinear"))
creates a model whose input is text. It has the usual .fit and .predict methods.
clone(estimator) copies settings into an unfitted estimator.

1. If chosen_name is "full" or "limited", construct **final_model** with
   make_pipeline(clone(the_chosen_vectorizer), clone(the_chosen_classifier)).
   If it is "baseline", use make_pipeline(CountVectorizer(), clone(baseline)).
2. Fit that fresh pipeline on development_data["review_clean"] and its sentiment
   labels. This combines training and validation data **after** your choice.
3. Obtain **final_predictions** on test_data["review_clean"]. Evaluate test
   accuracy and negative recall.
4. Separately fit **final_baseline**, a fresh pipeline containing CountVectorizer
   and DummyClassifier(strategy="most_frequent"), on the same development data.
   Obtain **final_baseline_predictions** on test text.
5. Display **final_results**, comparing those two approaches on test accuracy
   and negative recall. Draw a labelled test confusion matrix for final_model.

Relearning the full vocabulary on development text is now appropriate.
The fixed 20-word vocabulary stays fixed if selected. Neither approach learns
from test text or test labels. Do not change your choice after seeing test scores.


In [ ]:
# Your code: create fresh pipelines, refit on development text, evaluate on test reviews, and plot the final confusion matrix.


In [ ]:
required = ["chosen_name", "final_model", "final_predictions", "final_baseline", "final_baseline_predictions", "final_results"]
assert all(name in globals() for name in required), "Complete the choice, fresh refit, and final assessment."
assert chosen_name in {"baseline", "full", "limited"}, "Use a stated approach."
assert np.asarray(final_predictions).shape == (len(test_data),), "Obtain one label per test review."
assert np.array_equal(final_predictions, final_model.predict(test_data["review_clean"])), "Check the final text pipeline and test row order."
assert len(final_results) == 2, "Compare the chosen model and a development-fitted baseline."
print("Checkpoint passed: final predictions and the test comparison are ready.")


## Final reflection

Report the selected approach, its validation accuracy and negative recall before
refitting, and its final test accuracy and negative recall. Compare with the test
baseline. Why need the validation and test scores not match?

State two limits of this model or evaluation. Include one about the text/labels
and one about which future reviews this experiment can represent. Propose one
small extension and explain how you would compare it without repeatedly using
these test scores to choose the winner.


_Your answer:_


### Submission checklist

- Complete all core code blocks, plots, the model choice, and written answers.
- Keep the supplied groups and fixed model settings; use every eligible review.
- Label probabilities, proportions, percentages, and percentage-point differences clearly.
- Support comparisons with validation evidence and report negative-review errors.
- Restart the kernel and run the completed core notebook in order.
- Submit the completed notebook with useful tables and plots retained.

Optional work is not required for a complete core submission.


## Optional extensions

### A. Change the decision threshold — validation experiment only

The binary classifier normally predicts +1 above a positive probability of 0.5
(an exact tie may be resolved by label order). A different threshold changes
decisions **without refitting** the coefficients.

**Your turn:** using full_p_positive and the same validation labels, write a
loop for thresholds **0.2, 0.5, and 0.8**. For each, predict +1 when
P(+1) >= threshold and −1 otherwise. Build **threshold_results** with accuracy,
negative recall, and **negative precision**.

Negative precision asks: of all reviews we flagged as negative, what share
actually had a negative label? Use precision_score(..., pos_label=-1,
zero_division=0); zero_division=0 records 0 if no negative predictions exist.
Construct a line plot of negative precision and recall against threshold.

Keep the completed core test result and its standard rule unchanged. This is
a validation exploration, not an invitation to tune using test errors.


In [ ]:
# Your code: loop over thresholds, construct threshold_results, and plot negative precision and recall.


**Explain:** Which direction of threshold change flags more reviews as negative?
What happens to negative recall and precision in your results? Which might a
complaint-screening team prefer if staff time is limited? Would any particular
threshold be universally best?


_Your answer:_


### B. Try short texts and expose a representation limit

**Your turn:** create at least four short reviews, including the exact pair
"not good" and "good not", one example containing love, and one containing
disappointed. Clean them with the same rule as training. Use the fitted
**full_vectorizer and full_model** to build a table of labels and positive
probabilities. Do not fit on your examples.

Check whether the count vectors for the first pair are identical. Interpret
the predictions cautiously: these artificial snippets are a demonstration,
not a representative accuracy test, and no labelled evaluation set is supplied.


In [ ]:
# Your code: create and clean short texts, predict without fitting, and compare the reordered pair's vectors.


**Explain:** What does the reordered pair demonstrate? What would you need
before treating success on your own short examples as evidence of useful
performance on other people's reviews?


_Your answer:_


## Optional reference links

- [CountVectorizer and its tokenisation settings](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [LogisticRegression and probability output](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [DummyClassifier baselines](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)
- [Confusion-matrix row and column conventions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [Consistent preprocessing and data leakage](https://scikit-learn.org/stable/common_pitfalls.html)
